# 📈 E-Commerce Sales Forecasting: End-to-End Time-Series System
**Goal:** Predict future retail sales based on historical transaction data using **ARIMA/SARIMAX**, **Facebook Prophet**, **LSTM Neural Networks**, and **Machine Learning Ensemble Models**.

---

### 📋 Project Guidelines & Objectives:
1. **Time-Series Data**: Ingest and process e-commerce transactions across 326 continuous days and 7 product categories.
2. **Multi-Model Paradigm**:
   - **ARIMA / SARIMAX**: Classical statistical autoregression with exogenous regressors.
   - **Facebook Prophet**: Generalized additive model with changepoint detection and decomposed seasonalities.
   - **LSTM (Deep Learning)**: Long Short-Term Memory recurrent neural network capturing non-linear sequential dependencies.
   - **XGBoost & Weighted Ensemble**: Feature-driven gradient boosting and blended meta-forecaster.
3. **Components Incorporated**:
   - **Trend**: Long-term directional drift and moving averages.
   - **Seasonality**: Day of week (7-day cyclical patterns) and monthly fluctuations.
   - **Promotions & Discounts**: Promotional intensity, discount elasticity, and promotional interaction terms.
4. **Visualizations**: Interactive and static plots comparing forecast trajectories vs actual sales, with 95% confidence intervals and residual diagnostics.


## 1. Import Libraries & Configure Environment

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configure plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
warnings.filterwarnings('ignore')

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('.'))

from src.data_loader import load_raw_data, aggregate_time_series, get_dataset_summary
from src.feature_engineering import build_full_feature_matrix, perform_seasonal_decomposition
from src.models.arima_model import SARIMAXForecaster
from src.models.prophet_model import ProphetForecaster
from src.models.lstm_model import LSTMForecaster
from src.models.xgboost_model import XGBoostForecaster
from src.models.ensemble_model import EnsembleForecaster
from src.evaluation import calculate_metrics, compare_models, calculate_residual_diagnostics


## 2. Load & Explore E-Commerce Dataset

In [ ]:
df_raw = load_raw_data('ecommerce_dataset_updated.csv')
print(f"Dataset Shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head()


In [ ]:
summary = get_dataset_summary(df_raw)
print("=== High-Level Business KPIs ===")
for k, v in summary.items():
    if isinstance(v, float):
        print(f" • {k.replace('_', ' ').title()}: Rs. {v:,.2f}" if 'revenue' in k or 'value' in k else f" • {k.replace('_', ' ').title()}: {v:.2f}")
    elif isinstance(v, dict):
        print(f" • {k.replace('_', ' ').title()}: {v}")
    else:
        print(f" • {k.replace('_', ' ').title()}: {v}")


## 3. Continuous Time-Series Resampling

In [ ]:
# Aggregate transactions into daily regular intervals
df_daily = aggregate_time_series(df_raw, freq='D')
print(f"Total continuous days: {len(df_daily)}")
print(f"Date range: {df_daily.index.min().strftime('%Y-%m-%d')} to {df_daily.index.max().strftime('%Y-%m-%d')}")
df_daily.head()


In [ ]:
# Plot historical daily sales and 7-day rolling average
plt.figure(figsize=(14, 5))
plt.plot(df_daily.index, df_daily['sales'], label='Daily Sales (Rs.)', color='#0284C7', alpha=0.6, linewidth=1.5)
plt.plot(df_daily.index, df_daily['sales'].rolling(7).mean(), label='7-Day Moving Average', color='#DC2626', linewidth=2.5)
plt.title('Daily E-Commerce Sales Revenue (2024)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales Revenue (Rs.)')
plt.legend()
plt.tight_layout()
plt.show()


## 4. Seasonal, Trend & Residual Decomposition

In [ ]:
decomp = perform_seasonal_decomposition(df_daily['sales'], period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
axes[0].plot(decomp['observed'], color='#0284C7')
axes[0].set_title('Observed Series', fontweight='bold')
axes[1].plot(decomp['trend'], color='#DC2626')
axes[1].set_title('Estimated Underlying Trend', fontweight='bold')
axes[2].plot(decomp['seasonal'], color='#7C3AED')
axes[2].set_title('7-Day Weekly Seasonality Component', fontweight='bold')
axes[3].scatter(decomp['residual'].index, decomp['residual'], color='#475569', s=8)
axes[3].set_title('Residual Stochastic Noise', fontweight='bold')

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 5. Promotional & Discount Elasticity Analysis

In [ ]:
disc_group = df_raw.groupby('Discount (%)').agg(
    orders=('User_ID', 'count'),
    revenue=('Final_Price(Rs.)', 'sum')
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.bar(disc_group['Discount (%)'], disc_group['orders'], color='#6366F1', alpha=0.6, width=2.5, label='Order Count (Volume)')
ax2.plot(disc_group['Discount (%)'], disc_group['revenue'], color='#F59E0B', linewidth=3, marker='o', label='Total Revenue (Rs.)')

ax1.set_xlabel('Discount Percentage (%)', fontweight='bold')
ax1.set_ylabel('Order Count', color='#6366F1', fontweight='bold')
ax2.set_ylabel('Total Revenue (Rs.)', color='#F59E0B', fontweight='bold')
plt.title('Promotional Elasticity: Discount % vs Order Volume & Revenue', fontsize=14, fontweight='bold')
plt.show()


## 6. Feature Engineering & Train/Test Split

In [ ]:
df_features = build_full_feature_matrix(df_daily, target_col='sales')
print(f"Total features created: {df_features.shape[1]}")
print("Feature columns:", df_features.columns.tolist())

test_days = 30
train_df = df_features.iloc[:-test_days].copy()
test_df = df_features.iloc[-test_days:].copy()

print(f"\nTraining observations: {len(train_df)} ({train_df.index[0].strftime('%Y-%m-%d')} to {train_df.index[-1].strftime('%Y-%m-%d')})")
print(f"Testing observations:  {len(test_df)} ({test_df.index[0].strftime('%Y-%m-%d')} to {test_df.index[-1].strftime('%Y-%m-%d')})")


## 7. Model Training, Evaluation & Benchmarking

In [ ]:
exog_cols = ['discount_rate', 'is_weekend']
eval_preds = {}

# 1. SARIMAX
print("Fitting SARIMAX...")
sarimax = SARIMAXForecaster(order=(1, 1, 1), seasonal_order=(1, 0, 1, 7))
sarimax.fit(train_df['sales'], exog=train_df[exog_cols])
s_pred = sarimax.predict(steps=test_days, exog_future=test_df[exog_cols], start_date=test_df.index[0])
eval_preds['SARIMAX'] = s_pred['yhat'].values

# 2. Facebook Prophet
print("Fitting Prophet...")
prophet = ProphetForecaster()
prophet.fit(train_df['sales'], exog=train_df[exog_cols])
p_pred = prophet.predict(steps=test_days, exog_future=test_df[exog_cols])
eval_preds['Prophet'] = p_pred['yhat'].values

# 3. LSTM
print("Fitting LSTM Recurrent Neural Network...")
lstm = LSTMForecaster(lookback=14, epochs=50, batch_size=16)
lstm.fit(train_df['sales'], exog=train_df[exog_cols], verbose=0)
l_pred = lstm.predict(steps=test_days, exog_future=test_df[exog_cols])
eval_preds['LSTM'] = l_pred['yhat'].values

# 4. XGBoost
print("Fitting XGBoost ML Regressor...")
xgb_model = XGBoostForecaster(n_estimators=120, max_depth=3)
xgb_model.fit(train_df, target_col='sales')
x_pred = xgb_model.predict(steps=test_days)
eval_preds['XGBoost'] = x_pred['yhat'].values

# 5. Weighted Ensemble
print("Blending Ensemble Forecaster...")
ens = EnsembleForecaster(weights={'Prophet': 0.40, 'SARIMAX': 0.30, 'LSTM': 0.30})
ens_pred = ens.blend_forecasts({'SARIMAX': s_pred, 'Prophet': p_pred, 'LSTM': l_pred})
eval_preds['Ensemble'] = ens_pred['yhat'].values

# Compute evaluation metrics
metrics_dict = {m: calculate_metrics(test_df['sales'].values, preds) for m, preds in eval_preds.items()}
leaderboard = compare_models(metrics_dict)
print("\n=== Model Performance Leaderboard ===")
display(leaderboard)


## 8. Forecast vs. Actual Holdout Comparison Plot

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(test_df.index, test_df['sales'], color='black', linewidth=2.5, marker='o', label='Actual Ground Truth')
plt.plot(test_df.index, eval_preds['SARIMAX'], linestyle='--', color='#10B981', label='SARIMAX')
plt.plot(test_df.index, eval_preds['Prophet'], linestyle='--', color='#8B5CF6', label='Prophet')
plt.plot(test_df.index, eval_preds['LSTM'], linestyle='--', color='#EC4899', label='LSTM')
plt.plot(test_df.index, eval_preds['Ensemble'], linewidth=2.5, color='#06B6D4', label='Weighted Ensemble')

plt.title(f'Model Holdout Forecast Comparison (Last {test_days} Days)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales Revenue (Rs.)')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 9. Future 30-Day Out-of-Sample Sales Forecast

In [ ]:
horizon = 30
full_prophet = ProphetForecaster().fit(df_features['sales'], exog=df_features[exog_cols])

future_dates = pd.date_range(start=df_features.index[-1] + pd.Timedelta(days=1), periods=horizon, freq='D')
future_exog = pd.DataFrame({
    'discount_rate': [float(df_features['discount_rate'].mean())] * horizon,
    'is_weekend': (future_dates.dayofweek >= 5).astype(int)
}, index=future_dates)

future_pred = full_prophet.predict(steps=horizon, exog_future=future_exog)

plt.figure(figsize=(14, 6))
plt.plot(df_features.index[-60:], df_features['sales'].iloc[-60:], color='#0284C7', label='Historical Sales (Last 60d)')
plt.plot(future_pred.index, future_pred['yhat'], color='#F59E0B', linestyle='--', linewidth=2.5, label='Future Forecast (Prophet)')
plt.fill_between(future_pred.index, future_pred['yhat_lower'], future_pred['yhat_upper'], color='#F59E0B', alpha=0.2, label='95% Confidence Interval')

plt.title(f'Future {horizon}-Day Sales Forecast Trajectory with 95% Confidence Band', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales Revenue (Rs.)')
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("\nProjected Total Revenue for Next 30 Days: Rs.", f"{future_pred['yhat'].sum():,.2f}")
